[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/17_dropout_solution.ipynb)

# 🟢 Solution: Implement Dropout

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `17_dropout.ipynb` first.

---
Implement **inverted dropout** as an `nnx.Module`.

**Training:** zero each element independently with probability `rate`, then
divide the survivors by `(1 - rate)`.

$$y_i = \frac{x_i \cdot m_i}{1 - p}, \qquad m_i \sim \text{Bernoulli}(1-p)$$

**Inference:** return `x` unchanged.

### Rules
- Signature: `MyDropout(rate, *, rngs)`
- `__call__(x, deterministic=False)`
- Draw the mask from the `dropout` rng stream: `self.rngs.dropout()`
- Each call must use a **fresh** key — two training calls must give different masks
- `rate == 0.0` must be an exact no-op
- `deterministic=True` returns `x` unchanged, with **no** scaling

### Why divide during training
The point is to keep $\mathbb{E}[y] = x$ so the network sees the same expected
activation magnitude in both modes:

$$\mathbb{E}[y_i] = (1-p)\cdot\frac{x_i}{1-p} + p \cdot 0 = x_i$$

The original 2014 paper scaled by $(1-p)$ at **test** time instead. "Inverted"
dropout moves that correction into training so the inference path is a plain
identity — which matters because inference runs far more often, and because it
means you can strip dropout entirely when exporting a model.

### The rng-stream part
`nnx.Rngs(params=0, dropout=1)` sets up **named streams**. Calling
`rngs.dropout()` returns a fresh key each time and advances the stream, so you
get a new mask per call without threading keys through your own code. Using
`rngs.params()` here would be wrong — it would consume the initialisation
stream.

### A gotcha you will hit
Advancing the stream **mutates** the module, so plain `jax.grad` refuses to
differentiate through a dropout call:

```
TraceContextError: Cannot mutate RngCount from a different trace level
```

Use NNX's own transforms instead — they know how to split the module's state
out and thread it through functionally:

```python
g = nnx.grad(lambda m, v: jnp.sum(m(v)), argnums=1)(drop, x)
```

The same applies to `nnx.jit` over anything holding rngs or `BatchStat`.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class MyDropout(nnx.Module):
    def __init__(self, rate: float, *, rngs: nnx.Rngs):
        self.rate = rate
        self.rngs = rngs

    def __call__(self, x, deterministic: bool = False):
        if deterministic or self.rate == 0.0:
            return x

        keep_prob = 1.0 - self.rate
        # A fresh key per call — the stream advances automatically.
        key = self.rngs.dropout()
        keep = jax.random.bernoulli(key, keep_prob, x.shape)
        # Dividing here is what makes it "inverted": inference stays an identity.
        return jnp.where(keep, x, 0.0) / keep_prob

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

drop = MyDropout(0.5, rngs=nnx.Rngs(dropout=0))
x = jnp.ones((4, 6))

print("train call 1:\n", drop(x))
print("train call 2 (different mask):\n", drop(x))
print("eval:\n", drop(x, deterministic=True))
print("\nexpected value preserved:", float(jnp.mean(drop(jnp.ones((2000,))))), "(~1.0)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("dropout")